# 01 — Retrieval: index, search and evaluation

This notebook builds (or loads) the search index and **measures retrieval quality**. The reusable code lives in `src/`, so the next notebook (generation) and later the FastAPI app import exactly the same functions:

| Module | What it does |
|---|---|
| `src/config.py` | Paths, settings (chunk size, models, `TOP_K`) and the API key from `.env` |
| `src/ingest.py` | PDFs → pages → `clean_text` → `chunk_text` → chunks |
| `src/retrieval.py` | `Retriever` (e5 embeddings + FAISS index, save/load, search), `Reranker` (experiment) |
| `src/build_index.py` | Script: `python -m src.build_index` builds `index/` from `data/raw/` |
| `src/eval_retrieval.py` | Evidence-phrase evaluation: `hit@k`, `doc_hit@k`, `first_rank`, comparisons |
| `tests/` | Unit tests for cleaning, chunking and the evaluator: `python -m pytest` |

```
data/raw/*.pdf → load → clean → chunk → embed (e5) → FAISS index (index/) → search → [LLM: next notebook]
```

**How to run:** *Run All*. The first time, the index is built (~2 min on CPU, faster on GPU) and saved to `index/`; after that it loads in seconds. The reranker experiment (section 5) is switched off by default because it downloads a 2.3 GB model.

## 0. Setup

`%autoreload 2` makes the notebook pick up edits in `src/` without restarting the kernel.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")   # so that `import src...` works from notebooks/

import pandas as pd

from src.config import INDEX_DIR, N_CANDIDATES, TOP_K
from src.eval_retrieval import (compare_results, evaluate_retrieval, load_questions,
                                summarize, validate_phrases)
from src.ingest import build_chunks, load_pages, load_sources
from src.retrieval import Reranker, Retriever, get_device, make_rerank_search, time_search

DEVICE = get_device()
print("Device:", DEVICE, "| TOP_K:", TOP_K)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


W1107 23:26:30.892000 9044 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda | TOP_K: 15


## 1. Build or load the index

Embedding every chunk is the slow step, so it is done **once** and saved to `index/` (ignored by Git):

- `chunks.faiss` — the vectors in a FAISS `IndexFlatIP` (exact inner-product search; with normalized vectors, inner product = cosine similarity)
- `chunks.json` — the chunk texts and their filename/page
- `meta.json` — the model and chunk settings used. `Retriever.load` refuses an index built with different settings, because a stale index silently gives wrong results.

Set `REBUILD = True` after changing the documents, `clean_text`, chunking or the embedding model. (Equivalent from PowerShell: `python -m src.build_index`.)

In [4]:
REBUILD = False

if REBUILD or not (INDEX_DIR / "meta.json").exists():
    chunks = build_chunks(load_pages())
    retriever = Retriever.build(chunks, device=DEVICE)
    retriever.save()
    print("Built and saved a new index")
else:
    retriever = Retriever.load(device=DEVICE)
    print("Loaded index from", INDEX_DIR)

chunks = retriever.chunks
sources = load_sources()
print(retriever.index.ntotal, "vectors |", len(chunks), "chunks |", len(sources), "documents")

Batches: 100%|██████████| 12/12 [00:08<00:00,  1.48it/s]

Built and saved a new index
179 vectors | 179 chunks | 8 documents


## 2. Inspect the chunks

Chunks are built **within each page**, 1,000 characters with 200 overlap, ending at word boundaries (design notes in `src/ingest.py`).

The **token check** matters: e5 reads at most 512 tokens and silently truncates anything longer. Greek averages about 3.3 characters per token here.

In [5]:
df_chunks = pd.DataFrame(chunks)
print(df_chunks["text"].str.len().describe(), "\n")
print(df_chunks.groupby("filename").size(), "\n")

token_counts = [len(retriever.model.tokenizer("passage: " + c["text"])["input_ids"]) for c in chunks]
print("Max tokens per chunk:", max(token_counts), "/", retriever.model.max_seq_length)
assert max(token_counts) <= retriever.model.max_seq_length, "some chunks are truncated"

count    179.000000
mean     848.759777
std      252.961234
min       70.000000
25%      809.500000
50%      993.000000
75%      997.000000
max      999.000000
Name: text, dtype: float64 

filename
aade_pol1187_2017.pdf                  30
aade_str_faq_2025.pdf                  70
aade_str_guide_2024.pdf                26
aade_takk_faq_2026.pdf                 14
fek_A1202_2024.pdf                     12
govgr_str_restrictions_2026.pdf         2
law_5170_2025_art3.pdf                 13
mintour_str_specs_circular_2025.pdf    12
dtype: int64 

Max tokens per chunk: 448 / 512


## 3. Manual search

`show_top` prints the top results with their cosine similarity and a citation built from `sources.csv`. Useful to read what the system actually returns for a question.

In [6]:
def show_top(question: str, k: int = 5, search=None, preview: int = 250) -> None:
    """Print the top-k chunks for a question. `search` defaults to e5; pass another SearchFn to compare."""
    if search is None:
        results = retriever.search_with_scores(question, k)
    else:
        results = [(i, float("nan")) for i in search(question, k)]
    for rank, (idx, score) in enumerate(results, start=1):
        c = chunks[idx]
        title = sources.loc[c["filename"], "title"]
        print(f"#{rank}  score={score:.3f}  {title} — σελ. {c['page']}")
        print("    ", c["text"][:preview], "\n")


show_top("Μέχρι πότε πρέπει να δηλώσω μια μίσθωση στην ΑΑΔΕ;")

#1  score=0.878  Συχνές ερωτήσεις – απαντήσεις για τη Βραχυχρόνια Μίσθωση Ακινήτων — σελ. 9
     της 31ης Δεκεμβρίου 2024) και μία για το χρονικό διάστημα από 1.1.2025 έως 5.1.2025 αντίστοιχα, , αναγράφοντας στο πεδίο «Σημειώσεις» της κάθε δήλωσης ότι αφορά μέρος μίσθωσης με ημερομηνία αναχώρησης 5.1.2025. Στην περίπτωση αναχώρησης του μισθωτή σ 

#2  score=0.870  Συχνές ερωτήσεις – απαντήσεις για τη Βραχυχρόνια Μίσθωση Ακινήτων — σελ. 15
     ορίζεται μεταξύ της ημερομηνίας της τελευταίας βραχυχρόνιας μίσθωσης έως την ημερομηνία έναρξης υποβολής των δηλώσεων φορολογίας εισοδήματος του αντίστοιχου φορολογικού έτους. 41. Ε Πώς αποκτούν Αριθμό Φορολογικού Μητρώου (ΑΦΜ) τα αλλοδαπά πρόσωπα (Φ 

#3  score=0.868  Συχνές ερωτήσεις – απαντήσεις για τη Βραχυχρόνια Μίσθωση Ακινήτων — σελ. 2
     την περίπτωση θεωρείται πλέον μακροχρόνια και υφίσταται υποχρέωση υποβολής Δήλωσης Πληροφοριακών Στοιχείων Μίσθωσης Ακίνητης Περιουσίας σύμφωνα με τα όσα ορίζονται στην ΠΟΛ 1162/2018 (Β΄3579) β) Η μίσθω

## 4. Retrieval evaluation (e5)

`eval/retrieval_questions.csv`: 24 answerable questions, each with an **evidence phrase** (regex, alternatives separated by `|`) found only in chunks that answer it.

- **`hit@k`**: a chunk containing the evidence phrase is in the top k (chunk level)
- **`doc_hit@k`**: a chunk from an expected document is in the top k (document level; easier, and it hides chunk-level misses)
- **`first_rank`**: where the first correct chunk is, within the top 50

Each question is searched **once**; all k values are computed from the same ranking. With 24 questions, one question ≈ 4 percentage points.

In [7]:
questions = load_questions()
print("Phrases with no match:", validate_phrases(chunks, questions))

results_e5 = evaluate_retrieval(retriever.search, chunks, questions, ks=(5, 10, 15, 20))
summarize(results_e5)

Phrases with no match: []


,hit,doc_hit
k,,
5,0.625000,0.958333
10,0.750000,1.000000
15,0.833333,1.000000
20,0.916667,1.000000


**Misses at `TOP_K`** (the number of chunks the LLM will receive):

In [8]:
results_e5.loc[~results_e5[f"hit@{TOP_K}"], ["id", "first_rank"]]

,id,first_rank
0,REG-01,18
1,REG-02,28
9,BKG-04,41
22,RES-03,19


## 5. Experiment: cross-encoder reranker (GPU, optional)

e5 is a **bi-encoder**: question and chunk are embedded separately, which is fast but imprecise (scores of the top 20 differ by ~0.02). A **cross-encoder** reads question and chunk *together* and scores whether the chunk answers the question:

```
e5 top 30 → bge-reranker-v2-m3 re-scores the 30 → top 5
```

**Result:** hit@5 rose from 0.625 to 0.833 (6 fixed, 1 broken), but it costs ~3.9 s per question on a GTX 1650 Ti and is far too slow on CPU (a 2-vCPU Hugging Face Space would need about a minute per question). e5 top-15 reaches the same hit rate (0.833) at no extra cost, so **the deployed app uses e5 with `TOP_K = 15`** and the reranker stays a documented experiment.

Set `RUN_RERANKER = True` to reproduce it (downloads ~2.3 GB the first time).

In [9]:
RUN_RERANKER = False

if RUN_RERANKER:
    reranker = Reranker(device=DEVICE)            # fp16 on GPU
    search_rerank = make_rerank_search(retriever, reranker, n_candidates=N_CANDIDATES)

    results_rr = evaluate_retrieval(search_rerank, chunks, questions, ks=5, progress=True)
    print(summarize(results_rr))

    comparison = compare_results(results_e5, results_rr, name_a="e5", name_b="rerank", k=5)
    print(comparison["change"].value_counts())
    display(comparison)

**Latency** (average seconds per query, after a warm-up call; `time_search` calls `torch.cuda.synchronize()` so asynchronous GPU work is counted).

In [10]:
if RUN_RERANKER:
    sample = questions["question"].head(5).tolist()
    print(f"e5:     {time_search(retriever.search, sample):.3f} s/query")
    print(f"rerank: {time_search(search_rerank, sample):.3f} s/query on {DEVICE}")

## 6. Findings and decision

| Variant | hit@5 | hit@10 | hit@15 | hit@20 | s/query |
|---|---|---|---|---|---|
| Word overlap (reference) | 0.500 | | | | — |
| e5-base | 0.625 | 0.750 | **0.833** | 0.917 | fast |
| e5 top 30 + bge-reranker-v2-m3 | **0.833** | | | | ~3.9 (GPU), CPU too slow |

- **Decision:** e5 with `TOP_K = 15` chunks to the LLM. Whether the LLM still answers correctly when the right chunk is, say, 11th of 15 is measured with answer correctness in Milestone 3.
- **Right document, wrong chunk:** `doc_hit` is ~1.0 while `hit` is not; a document-level metric alone would hide this.
- **Scores don't separate answerable from unanswerable questions** (0.879 vs 0.823), so refusal is decided by the LLM prompt.
- **The corpus contains superseded rules** (old ΤΑΚΚ amounts in the 2024 guide, old deadline in ΠΟΛ 1187/2017): the prompt must prefer the newest source using `published_date`.
- **Still to check:** evidence phrases of REG-01, REG-04, TAX-04 may be too narrow (label audit); BKG-04's correct chunk is at e5 rank 41.

Full log: `eval/experiments.md`.